# Workflow 5: Parallel | Essay Moderator
Author: Nihit Kumar

---
**Task:** Moderate essay based on multiple points - Clarity of Thought (COA), Depth of Analysis (DOA) and Language (Grammer).  
Each task will be performed by an LLM and each of them will generate two outputs - score and feeback.  
Final Analysis - Combining all 3 scores, get average score. Create a summarizzed feedback by generating a new feedback (LLM-baased) by combining all feedbacks.  

**States:**
- `essay`: user input
- `score`: LLM Generated. Using reducer to avoid overwritting
- `doa_feedback`: LLM Generated
- `lang_feedback`: LLM Generated
- `summary`: LLM Generated
- `average_score`: LLM Generated  

---
**Reducer Function**: It is used where we need to add the values to a single state variable without actually removing the older variable.  
*operator* is used while defining the variuable in state along with annotated list. This ensures, that all values are not overwritten when a state is updated, instead it gets added as a list.  
This will be used in storing the list of scores in the state for each analysis.

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator

In [2]:
load_dotenv()

# We have to use a model which will support structured output
model = ChatOpenAI(model='gpt-4o-mini')

We are using the concept of structured output so all the LLMs can provide the output in desired format

In [3]:
# Creating an output schemaa for each evaluation criteria for LLMs
class EvaluationSchema(BaseModel):
    feedback: str = Field(description='Detailed feedback of the essay')
    score: int = Field(description='Provide a score out  of  10', ge=0, le=10)


In [4]:
updated_model = model.with_structured_output(EvaluationSchema)

In [5]:
## Essay for evaluation (also created from ChatGPT..:)
essay = f"""
India’s role in the modern AI revolution is both rapidly evolving and strategically significant. While it may not yet rival the foundational model-building dominance of countries like the United States or China, India has emerged as a powerful force in AI adoption, talent development, and applied innovation—areas that are increasingly shaping how AI impacts the real world.
At the core of India’s AI strength is its vast and growing pool of technical talent. With millions of engineers and one of the largest STEM graduate pipelines globally, India has become a major hub for AI development and deployment. Indian engineers and researchers contribute extensively to global AI projects, both within multinational companies and through open-source ecosystems. Companies like Google, Microsoft, and Amazon have established large research and development centers in India, leveraging this talent to build and scale AI-driven solutions.
Another defining aspect of India’s role is its focus on applied AI at scale. Unlike regions primarily focused on cutting-edge research, India has excelled in deploying AI to solve real-world challenges across sectors such as healthcare, agriculture, finance, and governance. For example, AI-powered diagnostics are being used to improve access to healthcare in rural areas, while precision agriculture tools help farmers optimize crop yields. In financial services, AI is enabling fraud detection, credit scoring, and digital payments at an unprecedented scale.
Government initiatives have also played a crucial role. Programs like Digital India and the National AI Strategy have laid the groundwork for integrating AI into public infrastructure. India’s emphasis on building digital public goods—such as identity systems, payment platforms, and data-sharing frameworks—creates a unique environment where AI can be deployed efficiently and inclusively. This approach positions India not just as a technology user, but as a model for scalable digital transformation.
Startups are another key driver of India’s AI momentum. The country has witnessed a surge in AI-focused startups working on everything from natural language processing in regional languages to computer vision applications for logistics and retail. This innovation ecosystem is supported by increasing venture capital investment and a growing culture of entrepreneurship.
However, India’s AI journey is not without challenges. Issues such as data quality, infrastructure gaps, regulatory uncertainty, and the need for advanced research capabilities remain significant hurdles. Additionally, ensuring ethical AI deployment—particularly in a diverse and populous country—requires careful consideration of bias, privacy, and accessibility.
In conclusion, India’s role in the modern AI revolution is defined less by dominance in foundational breakthroughs and more by its ability to democratize and scale AI solutions. By combining a strong talent base, practical innovation, and supportive policy frameworks, India is shaping how AI is used in everyday life. As the global AI landscape continues to evolve, India is poised to play an increasingly influential role—not just as a participant, but as a leader in making AI accessible and impactful for billions.
"""

In [6]:
# Validating the output schema

prompt = f"Evaluate the language of the following essay and provide a feedback and assign a score out of 10 \n {essay}"

updated_model.invoke(essay).score


8

In [7]:
# Creating state class
class EssayEvaluationState(TypedDict):
    essay: str
    cot_feedback: str
    doa_feedback: str
    lang_feedback: str
    summary: str
    score: Annotated[list[int], operator.add] # Using reducer function
    average_score: float

    # 'operator.add' can be treated as [5]+[6]+[7] resulting in [5,6,7]

In [8]:
def analyze_cot(state: EssayEvaluationState):
    prompt = f"Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}"
    result = updated_model.invoke(prompt)
    return {'cot_feedback': result.feedback, 'score':[result.score]}

In [9]:
def analyze_doa(state: EssayEvaluationState):
    prompt = f"Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}"
    result = updated_model.invoke(prompt)
    return {'doa_feedback': result.feedback, 'score':[result.score]}

In [10]:
def analyze_lang(state: EssayEvaluationState):
    prompt = f"Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}"
    result = updated_model.invoke(prompt)
    return {'lang_feedback': result.feedback, 'score':[result.score]}

In [11]:
def summarize(state: EssayEvaluationState):

    # Creating a summary feedback

    prompt = f"Create a detailed summary from the feedbacks mentioned below \n\nLanguage: {state['lang_feedback']} \n\nClarity Of thought {state['cot_feedback']} \n\nDepth of Analysis {state['doa_feedback']} \n"

    result = model.invoke(prompt)

    # Average Score
    avg = sum(state['score']) / len(state['score'])

    return {'summary': result.content, 'average_score': avg}

In [12]:
graph = StateGraph(EssayEvaluationState)

graph.add_node('analyze_cot',analyze_cot)
graph.add_node('analyze_doa', analyze_doa)
graph.add_node('analyze_lang',analyze_lang)
graph.add_node('summarize',summarize)

graph.add_edge(START,'analyze_cot')
graph.add_edge(START,'analyze_doa')
graph.add_edge(START,'analyze_lang')

graph.add_edge('analyze_cot','summarize')
graph.add_edge('analyze_doa','summarize')
graph.add_edge('analyze_lang','summarize')

graph.add_edge('summarize', END)

workflow = graph.compile()

In [13]:
initial_state = {
    'essay' : essay
}

final_state = workflow.invoke(initial_state)

In [14]:
print(final_state['summary'])

The essay presents a comprehensive exploration of India’s evolving role in the artificial intelligence (AI) revolution. It effectively communicates the significance of India in the global AI landscape by establishing a clear context and main argument within the introduction. The use of a comparative perspective effectively highlights India’s unique position, and the logical flow of ideas is maintained through well-structured paragraphs that transition smoothly, supporting the overarching thesis.

The essay successfully employs specific examples to illustrate the applications of AI in critical sectors such as healthcare and agriculture, adding credibility to the argument. Furthermore, it discusses government initiatives and the dynamic startup ecosystem, showcasing the multifaceted approach India is taking to develop its AI capabilities.

Despite its strengths, the essay could enhance its discussion of the challenges faced by India in the AI sector. While some difficulties are acknowled